# Diabetes Risk Predictor: A Mini ML Project

This notebook aims to develop a machine learning model to predict diabetes risk based on health indicators. We will be using the 'Diabetes Health Indicators Dataset' from Kaggle.

## Setup Kaggle API for Dataset Download
To download the dataset directly from Kaggle, you'll need a Kaggle API key. Follow these steps:
1.  Go to Kaggle.com and log in.
2.  Click on your profile picture (top right) and select 'Your Profile'.
3.  Navigate to the 'Account' tab.
4.  Scroll down to the 'API' section and click 'Create New API Token'. This will download a `kaggle.json` file to your computer.
5.  Open the `kaggle.json` file. It contains your username and API key.
6.  In Colab, go to the 'Secrets' tab (🔑 icon on the left panel).
7.  Add a new secret. For the name, use `KAGGLE_USERNAME` and paste your Kaggle username as the value. Do the same for `KAGGLE_KEY`, pasting your API key as the value. Make sure 'Notebook access' is enabled for both.

### Target Variable Distribution

Let's examine the distribution of our target variable, `Diabetes_binary`, to understand if we have a balanced dataset.

In [ ]:
# Check the distribution of the target variable
display(df['Diabetes_binary'].value_counts())
display(df['Diabetes_binary'].value_counts(normalize=True))

### Feature Distributions and Correlations

To get a better sense of our features, let's visualize the distribution of a few key numerical features and examine the correlation matrix.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for the plots
sns.set_style("whitegrid")

# Plot histograms for selected numerical features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['BMI'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of BMI')

sns.histplot(df['Age'], kde=True, ax=axes[1])
axes[1].set_title('Distribution of Age (1-13 scale)')

sns.histplot(df['MentHlth'], kde=True, ax=axes[2])
axes[2].set_title('Distribution of Mental Health Days (MentHlth)')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate the correlation matrix
corr_matrix = df.corr()

# Plot the correlation matrix as a heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Features')
plt.show()

In [1]:
# Install the Kaggle API client
%pip install kaggle

In [2]:
# Set up Kaggle API credentials from Colab secrets
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

SecretNotFoundError: Secret KAGGLE_USERNAME does not exist.

In [3]:
# Download the dataset
!kaggle datasets download -d alexteboul/diabetes-health-indicators-dataset

# Unzip the dataset
import zipfile
import os

zip_file_path = 'diabetes-health-indicators-dataset.zip'
if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall('./data')
    print('Dataset unzipped successfully to ./data directory.')
else:
    print(f'Zip file not found at {zip_file_path}. Please ensure the download was successful.')

Dataset URL: https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset
License(s): CC0-1.0
100% 6.03M/6.03M [00:00<00:00, 97.9MB/s]

Dataset unzipped successfully to ./data directory.


In [4]:
import pandas as pd

# Load the dataset into a pandas DataFrame
df = pd.read_csv('./data/diabetes_binary_health_indicators_BRFSS2015.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## Exploratory Data Analysis (EDA)

Let's start by examining the dataset's general information and checking for any missing values.

In [5]:
# Display basic information about the DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_binary       253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  

In [6]:
# Check for missing values
display(df.isnull().sum())

,0
Diabetes_binary,0
HighBP,0
HighChol,0
CholCheck,0
BMI,0
Smoker,0
Stroke,0
HeartDiseaseorAttack,0
PhysActivity,0
Fruits,0
